# Homework 4 — Evaluation Metrics for Classification# Tarefa 4 — Métricas de Avaliação para Classificação**ML Zoomcamp 2026 — DataTalks.Club****Módulo 4** · Prazo: segunda, 19/10/2026, 20:00**Autor:** Nicolau Calado Jofilsan**Notebook bilíngue PT/EN** · `github.com/nicolaucalado/mlzoomcamp-2026`---## Ideia central / Core idea**PT:** Acurácia sozinha engana. Em uma base com 5% de positivos, um modelo que responde "não" para tudo acerta 95%. Toda a discussão de métricas existe para responder a três perguntas: que tipo de erro custa mais?, em que limiar eu decido? e a métrica vale para todos os grupos?**EN:** Accuracy alone is misleading. On a dataset with 5% positives, a model that always answers "no" is correct 95% of the time. The whole discussion of metrics exists to answer three questions: which type of error costs more?, at which threshold do I decide? and does the metric hold for all groups?> 🎯 **Seu diferencial:** este notebook conecta as métricas com o seu artigo IEEE Access — **vazamento de dados por participante (data leakage)**, onde uma acurácia de 83,5% por amostra cai para 51,2% quando a validação respeita os grupos (jogadores).

## 1. Setup e dados## 1. Setup and dataCarregamos bibliotecas e criamos um dataset sintético de classificação binária (churn / não-churn de clientes de um serviço digital). Substitua pelo seu dataset real (ex.: Olist, dados da tese) quando aplicar nos projetos P1/P2/P3.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_score, GroupKFoldfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    confusion_matrix, accuracy_score, precision_score, recall_score,    f1_score, roc_curve, auc, precision_recall_curve, average_precision_score,    classification_report)from sklearn.calibration import calibration_curvenp.random.seed(42)sns.set_style("whitegrid")# Dataset sintetico: 2000 clientes, 12% de churn (desbalanceado)n = 2000X = pd.DataFrame({    "uso_mes": np.random.normal(15, 5, n),    "tickets_suporte": np.random.poisson(2, n),    "idade": np.random.normal(35, 12, n),    "renda": np.random.normal(5000, 1500, n),})# Churn depende de uso baixo, muitos tickets e idade baixalogit = -3 + 0.4 * (15 - X["uso_mes"]) + 0.3 * X["tickets_suporte"] - 0.02 * X["idade"]prob = 1 / (1 + np.exp(-logit))y = (np.random.random(n) < prob).astype(int)df = pd.DataFrame({"cliente_id": [f"C{i:04d}" for i in range(n)]})df = pd.concat([df, X], axis=1)df["churn"] = yprint(f"Total de amostras: {len(df)}")print(f"Taxa de churn (positivos): {df['churn'].mean():.1%}")print(df.head())

## 2. Matriz de confusao / Confusion matrix**PT:** Quatro contagens, a partir de um limiar de decisao (padrao 0,5). VP, VN, FP, FN.**EN:** Four counts, given a decision threshold (default 0.5). TP, TN, FP, FN.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    df[["uso_mes", "tickets_suporte", "idade", "renda"]],    df["churn"],    test_size=0.3,    random_state=42,    stratify=df["churn"],)model = LogisticRegression(max_iter=1000)model.fit(X_train, y_train)y_pred = model.predict(X_test)y_proba = model.predict_proba(X_test)[:, 1]cm = confusion_matrix(y_test, y_pred)print("Matriz de confusao (sklearn):")print(cm)print()tn, fp, fn, tp = cm.ravel()print(f"TP (verdadeiros positivos): {tp}")print(f"TN (verdadeiros negativos): {tn}")print(f"FP (falsos positivos): {fp}")print(f"FN (falsos negativos): {fn}")plt.figure(figsize=(5, 4))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",            xticklabels=["Previsto Negativo", "Previsto Positivo"],            yticklabels=["Real Negativo", "Real Positivo"])plt.title("Matriz de Confusao / Confusion Matrix")plt.ylabel("Real / Actual")plt.xlabel("Previsto / Predicted")plt.tight_layout()plt.show()

## 3. Metricas principais / Main metrics| Metrica | Formula | Quando usar ||---------|---------|-------------|| Acuracia / Accuracy | (TP+TN) / total | Classes equilibradas e custos de erro parecidos || Precisao / Precision | TP / (TP+FP) | Falso positivo custa caro (bloquear cliente legitimo) || Revocacao / Recall | TP / (TP+FN) | Falso negativo custa caro (doenca nao detectada) || F1 | 2·(P·R)/(P+R) | Precisa de um numero so e ambas importam || Especificidade / Specificity | TN / (TN+FP) | Complemento da revocacao; base do eixo X da ROC |

In [ ]:
accuracy = accuracy_score(y_test, y_pred)precision = precision_score(y_test, y_pred)recall = recall_score(y_test, y_pred)f1 = f1_score(y_test, y_pred)specificity = tn / (tn + fp)print("=== Metricas de classificacao / Classification metrics ===")print(f"Acuracia / Accuracy : {accuracy:.4f}")print(f"Precisao / Precision: {precision:.4f}")print(f"Revocacao / Recall  : {recall:.4f}")print(f"F1-score            : {f1:.4f}")print(f"Especificidade      : {specificity:.4f}")print()print("=== Relatório detalhado / classification_report ===")print(classification_report(y_test, y_pred, target_names=["Nao churn", "Churn"]))

## 4. Limiar, curva ROC e AUC / Threshold, ROC and AUC**PT:** Um classificador devolve uma probabilidade, nao uma classe. O limiar (0,5 padrao) transforma probabilidade em decisao. Mexer no limiar troca precisao por revocacao: nao existe almoço gratis, existe escolha de negocio.**EN:** A classifier outputs a probability, not a class. The threshold (0.5 default) transforms probability into decision. Moving the threshold trades precision for recall: there is no free lunch, only a business choice.**AUC** eh a area sob a curva ROC. Interpretacao util em entrevista: eh a probabilidade de o modelo dar uma nota maior a um positivo sorteado ao acaso do que a um negativo sorteado ao acaso. Nao depende do limiar. Em bases muito desbalanceadas, a AUC pode parecer otima enquanto a precisao e pessima — nesses casos, use a curva precisao-revocacao (PR AUC).

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)roc_auc = auc(fpr, tpr)precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)pr_auc = average_precision_score(y_test, y_proba)fig, axes = plt.subplots(1, 2, figsize=(12, 5))axes[0].plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})", linewidth=2)axes[0].plot([0, 1], [0, 1], "k--", label="Acaso / Random")axes[0].set_xlabel("Taxa de falso positivo / False Positive Rate")axes[0].set_ylabel("Taxa de verdadeiro positivo / True Positive Rate")axes[0].set_title("Curva ROC / ROC Curve")axes[0].legend(loc="lower right")axes[0].grid(True, alpha=0.3)axes[1].plot(recall_curve, precision_curve, label=f"PR curve (AUC = {pr_auc:.3f})", linewidth=2)axes[1].axhline(y=df["churn"].mean(), color="k", linestyle="--", label="Taxa base / Baseline")axes[1].set_xlabel("Revocacao / Recall")axes[1].set_ylabel("Precisao / Precision")axes[1].set_title("Curva Precisao-Revocacao / Precision-Recall Curve")axes[1].legend(loc="lower left")axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print(f"ROC AUC : {roc_auc:.4f}")print(f"PR AUC  : {pr_auc:.4f}")print(f"Taxa de churn na base (baseline) : {df['churn'].mean():.4f}")

## 5. Validacao: split, k-fold estratificado e GroupKFold / Validation**PT:** Split treino/validacao/teste: treino ajusta o modelo, validacao escolhe hiperparametros e limiar, teste eh usado uma vez, no fim. Reusar o teste para escolher eh vazar informacao.**K-fold estratificado** mantem a proporcao de classes em cada parte. Padrao em classificacao desbalanceada.**GroupKFold:** quando varias linhas vem da mesma pessoa, sessao, paciente ou dispositivo, todas precisam ficar do mesmo lado da divisao. Caso contrario o modelo reconhece o individuo, nao o fenomeno, e a metrica sobe sem que o modelo tenha aprendido a generalizar.> ⚠️ **Conexao com o seu artigo IEEE Access:** este eh exatamente o achado — avaliar por amostra (sem respeitar grupos) infla a metrica. Avaliar por participante (GroupKFold) revela a verdadeira capacidade de generalizacao.

In [ ]:
# Simulacao de dados agrupados por participante (ex: 50 jogadores, 40 amostras cada)n_groups = 50samples_per_group = 40group_ids = np.repeat(range(n_groups), samples_per_group)X_grouped = pd.DataFrame({    "feature_a": np.random.normal(0, 1, n_groups * samples_per_group),    "feature_b": np.random.normal(0, 1, n_groups * samples_per_group),})# O sinal esta no grupo (participante), nao apenas na featuregroup_effect = np.repeat(np.random.normal(0, 0.5, n_groups), samples_per_group)logit_g = 1 + X_grouped["feature_a"] * 0.5 + group_effectprob_g = 1 / (1 + np.exp(-logit_g))y_grouped = (np.random.random(n_groups * samples_per_group) < prob_g).astype(int)print(f"Total de linhas: {len(X_grouped)}")print(f"Numero de grupos (participantes): {n_groups}")print(f"Taxa positiva: {y_grouped.mean():.1%}")# 1. Validacao aleatoria (estratificada) - IGNORA os gruposcv_random = cross_val_score(    LogisticRegression(max_iter=1000),    X_grouped, y_grouped,    cv=5,    scoring="accuracy",)# 2. GroupKFold - RESPEITA os grupos (mesmo participante nao divide treino/teste)gkf = GroupKFold(n_splits=5)cv_group = cross_val_score(    LogisticRegression(max_iter=1000),    X_grouped, y_grouped,    groups=group_ids,    cv=gkf,    scoring="accuracy",)print()print("=== Comparacao de validacao / Validation comparison ===")print(f"K-fold estratificado (ignora grupos): {cv_random.mean():.4f} (+-{cv_random.std():.4f})")print(f"GroupKFold (respeita grupos)        : {cv_group.mean():.4f} (+-{cv_group.std():.4f})")print(f"Diferenca (inflacao por vazamento)  : {cv_random.mean() - cv_group.mean():.4f}")plt.figure(figsize=(7, 4))plt.boxplot([cv_random, cv_group],            labels=["K-fold aleatorio (vazamento)", "GroupKFold (por participante)"])plt.ylabel("Acuracia / Accuracy")plt.title("Impacto do vazamento de dados por participante / Participant-level leakage")plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 6. Calibracao / Calibration**PT:** Um modelo eh calibrado quando, entre os casos a que ele atribui 0,7, cerca de 70% sao de fato positivos. AUC alta nao implica calibracao. Importa sempre que a probabilidade vira decisao de negocio (preco, prioridade de fila, corte de risco).**EN:** A model is calibrated when, among cases it assigns 0.7, about 70% are truly positive. High AUC does not imply calibration. It matters whenever the probability becomes a business decision (price, queue priority, risk cutoff).

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)plt.figure(figsize=(6, 5))plt.plot(mean_pred, frac_pos, "s-", label="Modelo / Model")plt.plot([0, 1], [0, 1], "k:", label="Calibrado perfeitamente / Perfectly calibrated")plt.xlabel("Probabilidade prevista media / Mean predicted probability")plt.ylabel("Fracao de positivos / Fraction of positives")plt.title("Curva de calibracao / Calibration curve")plt.legend()plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 7. Como escolher o limiar? / How to choose the threshold?**PT:** Na validacao, nao no teste. Pelo custo de negocio ou por uma restricao operacional (ex.: capacidade diaria da equipe que trata os casos sinalizados).**EN:** On validation, never on the test set. Based on business cost or an operational constraint (e.g., daily capacity of the team handling flagged cases).

In [ ]:
# Exemplo: diferentes limiares e seu impacto em precisao e revocacaofor thr in [0.3, 0.5, 0.7]:    y_pred_thr = (y_proba >= thr).astype(int)    p = precision_score(y_test, y_pred_thr, zero_division=0)    r = recall_score(y_test, y_pred_thr, zero_division=0)    f = f1_score(y_test, y_pred_thr, zero_division=0)    print(f"Limiar {thr}: precisao={p:.3f} | revocacao={r:.3f} | F1={f:.3f}")

## 8. Perguntas de entrevista / Interview questions1. Seu modelo tem 95% de acuracia. Isso eh bom?   **R:** Depende da proporcao de classes e do custo dos erros. Peça a matriz de confusao e a taxa base: com 5% de positivos, 95% eh o resultado de prever sempre a classe majoritaria.2. Precisao ou revocacao?   **R:** Depende de qual erro custa mais. Triagem de doença: revocacao. Bloqueio automatico de conta: precisao. Em produto, traduzo isso em custo por FP e por FN e escolho o limiar que minimiza o custo total.3. O que eh AUC, em uma frase?   **R:** A probabilidade de o modelo ordenar um positivo acima de um negativo, ambos sorteados ao acaso. Mede ordenacao, nao decisao.4. Quando a AUC engana?   **R:** Com forte desbalanceamento, porque a taxa de falsos positivos tem denominador enorme. Uso PR AUC e olho precisao na faixa de operacao real.5. Como voce detecta vazamento de dados por participante?   **R:** Uso GroupKFold (ou LeaveOneGroupOut) para garantir que todas as linhas de um mesmo participante fiquem do mesmo lado da divisao. Comparo com validacao aleatoria — se a metrica cai muito, ha vazamento. Isso eh exatamente o que meu artigo IEEE Access demonstra.

## 9. Glossario rapido EN / Quick glossary ENtrue/false positive, true/false negative, precision, recall (sensitivity), specificity, F1 score, threshold, ROC curve, AUC, precision-recall curve, calibration, class imbalance, stratified k-fold, group k-fold, data leakage.---*Notebook bilíngue PT/EN — ML Zoomcamp 2026, Modulo 4 — github.com/nicolaucalado/mlzoomcamp-2026*